# Options Scanner v4 — Phase 1

**CALL-side scanner with quality scoring gate**

A signal is only generated when conviction score >= 6/10.
If nothing qualifies, the scanner tells you to wait.
Every signal includes exact entry, stop, target, and time stop.

---

| Score | Conviction | Action |
|-------|------------|--------|
| 9-10  | VERY HIGH  | Strong candidate |
| 7-8   | HIGH       | Good candidate |
| 6     | MODERATE   | Minimum threshold — smaller size |
| < 6   | NONE       | No signal — wait |

**Scoring breakdown (10 pts total):**
Regime (2) + RSI (2) + Trend (2) + Volume (2) + Weekly (1) + Support (1)

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy'])
print('Dependencies ready')

In [ ]:
# Cell 2 — Load scanner from GitHub
# Replace YOUR_USERNAME with kevin-mumaw
!wget -q https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py

from options_scanner_v4 import (
    run_scan, print_results, deep_dive,
    ALL_SYMBOLS, WATCHLIST, CONFIG
)
print('Scanner loaded')
print(f'Watchlist: {len(ALL_SYMBOLS)} symbols')
print('Groups:', list(WATCHLIST.keys()))

In [ ]:
# Cell 3 — Configure
# Edit account size here. Everything else auto-adjusts.

MY_CONFIG = {**CONFIG}  # copy defaults

MY_CONFIG['account_size'] = 2000.00  # update as your account grows

# Optional: scan only specific symbols instead of full watchlist
# MY_SYMBOLS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'JPM']
MY_SYMBOLS = None  # None = scan all 25 symbols

print(f'Account: ${MY_CONFIG["account_size"]:,.0f}')
print(f'Min score to generate signal: {MY_CONFIG["min_score"]}/10')
print(f'Stop loss: -{int(MY_CONFIG["stop_loss_pct"]*100)}% | '
      f'Profit target: +{int(MY_CONFIG["profit_target_pct"]*100)}% | '
      f'Time stop: {MY_CONFIG["time_stop_dte"]} DTE')

In [ ]:
# Cell 4 — Run the scan
# This is the main cell. Run this daily.
results = run_scan(symbols=MY_SYMBOLS, config=MY_CONFIG)
print_results(results)

In [ ]:
# Cell 5 — Deep dive on a specific symbol
# Use this to investigate any symbol in detail,
# whether it appeared in the scan or not.

SYMBOL = 'GOOGL'  # change this

deep_dive(SYMBOL, config=MY_CONFIG)

In [ ]:
# Cell 6 — Review all scores from last scan
# See every symbol's score ranked highest to lowest.

import pandas as pd

rows = []
for r in results['all_results']:
    if not r.get('error'):
        rows.append({
            'Symbol':    r['symbol'],
            'Score':     r['score'],
            'Conviction':r['conviction'],
            'Trend':     r.get('trend','—'),
            'RSI':       r.get('rsi','—'),
            'Volume':    r.get('vol_data',{}).get('label','—'),
            'Weekly':    r.get('weekly',{}).get('trend','—'),
            'Has Option':r.get('option') is not None,
        })

df = pd.DataFrame(rows).sort_values('Score', ascending=False)
print(df.to_string(index=False))